# RNA-seq Exploratory Analysis: Stress Response in Plant Transcriptomes

**Author:** Samuel Ukwueze  
**Date:** April 2026  

---

I put this together to explore how gene expression shifts under stress conditions using a simulated RNA-seq dataset. My approach is quite straightforward: take raw count data, normalize it, run differential expression, and see what pathways are lighting up. Nothing fancy, just clean and reproducible.

The dataset here is synthetic but structured to mirror what you'd realistically get from something like NCBI GEO for a plant stress study.

## 1. Setup and Imports

I'm using the usual stack; NumPy and Pandas for data handling, Matplotlib and Seaborn for plots, and SciPy for the stats side of things.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

print('Libraries loaded successfully.')

## 2. Simulating the Count Matrix

I'm generating count data for 500 genes across 6 samples: 3 control and 3 treated (stress condition). To keep it realistic, I'm pulling from a negative binomial distribution since that's what real RNA-seq count data looks like. I also deliberately introduced differential expression in about 10% of genes.

In [ ]:
n_genes = 500
n_control = 3
n_treated = 3

gene_ids = [f'Gene_{i+1:03d}' for i in range(n_genes)]
sample_ids = [f'Control_{i+1}' for i in range(n_control)] + [f'Treated_{i+1}' for i in range(n_treated)]

# Base expression drawn from negative binomial
base_mean = np.random.negative_binomial(5, 0.3, n_genes).astype(float) + 10

control_data = np.column_stack([
    np.random.negative_binomial(base_mean.astype(int), 0.5) for _ in range(n_control)
])

# Introduce fold-change in ~10% of genes for treated samples
de_genes_idx = np.random.choice(n_genes, size=50, replace=False)
fold_changes = np.ones(n_genes)
fold_changes[de_genes_idx] = np.random.choice([2.5, 3.0, 4.0, 0.25, 0.33], size=50)

treated_mean = (base_mean * fold_changes).astype(int)
treated_mean = np.clip(treated_mean, 1, None)

treated_data = np.column_stack([
    np.random.negative_binomial(treated_mean, 0.5) for _ in range(n_treated)
])

counts = np.hstack([control_data, treated_data])
df_counts = pd.DataFrame(counts, index=gene_ids, columns=sample_ids)

print(f'Count matrix shape: {df_counts.shape}')
df_counts.head(8)

## 3. Quality Check: Library Sizes

Before anything else, I always check library sizes. Uneven sequencing depth across samples is one of those things that will quietly ruin your downstream analysis if you don't catch it early.

In [ ]:
lib_sizes = df_counts.sum(axis=0)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#4C72B0'] * n_control + ['#DD8452'] * n_treated
bars = ax.bar(sample_ids, lib_sizes / 1e6, color=colors, edgecolor='white', linewidth=0.8)
ax.set_xlabel('Sample')
ax.set_ylabel('Library Size (millions of reads)')
ax.set_title('Library Size per Sample')
ax.tick_params(axis='x', rotation=30)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#4C72B0', label='Control'),
                   Patch(facecolor='#DD8452', label='Treated')]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

print(lib_sizes.to_string())

## 4. Normalization: CPM

I'm normalizing to CPM (Counts Per Million) here. It's not as robust as TMM or DESeq2's size factors for publication-level work, but for an exploratory analysis it gets the job done and keeps things transparent.

In [ ]:
cpm = df_counts.div(df_counts.sum(axis=0), axis=1) * 1e6
log_cpm = np.log2(cpm + 1)  # pseudocount of 1 to handle zeros

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

log_cpm.plot(kind='box', ax=axes[0], color='steelblue', patch_artist=True)
axes[0].set_title('log2(CPM + 1) Distribution per Sample')
axes[0].set_xlabel('Sample')
axes[0].set_ylabel('log2(CPM + 1)')
axes[0].tick_params(axis='x', rotation=30)

for col in log_cpm.columns:
    axes[1].hist(log_cpm[col], bins=40, alpha=0.5, label=col)
axes[1].set_title('Expression Density')
axes[1].set_xlabel('log2(CPM + 1)')
axes[1].set_ylabel('Gene Count')
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

## 5. PCA: Sample Clustering

PCA is my go-to sanity check after normalization. If control samples don't cluster together and treated samples don't cluster separately, something's off; either a batch effect, a mislabeled sample, or a normalization issue worth investigating.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = log_cpm.T.values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
pcs = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(pcs, columns=['PC1', 'PC2'], index=sample_ids)
pca_df['Condition'] = ['Control'] * n_control + ['Treated'] * n_treated

fig, ax = plt.subplots(figsize=(6, 5))
palette = {'Control': '#4C72B0', 'Treated': '#DD8452'}

for condition, group in pca_df.groupby('Condition'):
    ax.scatter(group['PC1'], group['PC2'], label=condition,
               color=palette[condition], s=120, edgecolors='white', linewidth=0.8)
    for idx, row in group.iterrows():
        ax.annotate(idx, (row['PC1'], row['PC2']), textcoords='offset points',
                    xytext=(6, 4), fontsize=8)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.set_title('PCA: Sample Separation by Condition')
ax.legend()
plt.tight_layout()
plt.show()

print(f'PC1 explains {pca.explained_variance_ratio_[0]*100:.1f}% of variance')
print(f'PC2 explains {pca.explained_variance_ratio_[1]*100:.1f}% of variance')

## 6. Differential Expression Analysis

I'm running Welch's t-test gene by gene between control and treated groups, then applying Benjamini-Hochberg correction for multiple testing. Thresholds I'm using: adjusted p-value < 0.05 and |log2FC| > 1.

In [ ]:
from statsmodels.stats.multitest import multipletests

control_cols = [c for c in df_counts.columns if 'Control' in c]
treated_cols = [c for c in df_counts.columns if 'Treated' in c]

results = []
for gene in log_cpm.index:
    ctrl_vals = log_cpm.loc[gene, control_cols].values
    trt_vals  = log_cpm.loc[gene, treated_cols].values
    t_stat, p_val = ttest_ind(trt_vals, ctrl_vals, equal_var=False)
    log2fc = np.mean(trt_vals) - np.mean(ctrl_vals)
    results.append({'gene': gene, 'log2FC': log2fc, 'pvalue': p_val})

de_results = pd.DataFrame(results).set_index('gene')

# BH correction
_, padj, _, _ = multipletests(de_results['pvalue'], method='fdr_bh')
de_results['padj'] = padj

# Classify genes
de_results['significant'] = (de_results['padj'] < 0.05) & (de_results['log2FC'].abs() > 1)
de_results['direction'] = 'Not significant'
de_results.loc[(de_results['significant']) & (de_results['log2FC'] > 0), 'direction'] = 'Upregulated'
de_results.loc[(de_results['significant']) & (de_results['log2FC'] < 0), 'direction'] = 'Downregulated'

summary = de_results['direction'].value_counts()
print('Differential Expression Summary:')
print(summary.to_string())
de_results.sort_values('padj').head(10)

## 7. Volcano Plot

Volcano plots are probably the most intuitive way to visualize DE results — fold change on the x-axis, statistical significance on the y-axis. Top genes labeled for quick reference.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

color_map = {'Not significant': '#AAAAAA', 'Upregulated': '#E74C3C', 'Downregulated': '#3498DB'}

for direction, group in de_results.groupby('direction'):
    ax.scatter(group['log2FC'], -np.log10(group['padj'] + 1e-10),
               c=color_map[direction], label=direction, alpha=0.7, s=25, edgecolors='none')

# Threshold lines
ax.axhline(-np.log10(0.05), color='black', linestyle='--', linewidth=0.8, alpha=0.6)
ax.axvline(1, color='black', linestyle='--', linewidth=0.8, alpha=0.6)
ax.axvline(-1, color='black', linestyle='--', linewidth=0.8, alpha=0.6)

# Label top 10 genes
top_genes = de_results[de_results['significant']].nsmallest(10, 'padj')
for gene, row in top_genes.iterrows():
    ax.annotate(gene, (row['log2FC'], -np.log10(row['padj'] + 1e-10)),
                fontsize=7, ha='left', xytext=(4, 2), textcoords='offset points')

ax.set_xlabel('log2 Fold Change (Treated / Control)')
ax.set_ylabel('-log10(Adjusted p-value)')
ax.set_title('Volcano Plot — Differential Expression Under Stress')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Heatmap: Top DE Genes

Finally, a heatmap of the top 30 differentially expressed genes. This gives a cleaner picture of expression patterns across individual samples.

In [ ]:
top30 = de_results[de_results['significant']].nsmallest(30, 'padj').index
heatmap_data = log_cpm.loc[top30]

# Z-score across samples for each gene
heatmap_z = heatmap_data.apply(lambda row: (row - row.mean()) / row.std(), axis=1)

fig, ax = plt.subplots(figsize=(9, 10))
sns.heatmap(heatmap_z, cmap='RdBu_r', center=0, linewidths=0.3,
            linecolor='white', ax=ax, cbar_kws={'label': 'Z-score'})
ax.set_title('Top 30 DE Genes: Z-scored Expression', fontsize=12)
ax.set_xlabel('Sample')
ax.set_ylabel('Gene')
plt.tight_layout()
plt.show()

## 9. Summary

The analysis held up pretty well. Samples separated cleanly on PCA, which tells me the simulated condition effect is strong enough to pick up. After BH correction, we ended up with a solid number of statistically significant DE genes at the thresholds I set.

A few things I'd do differently with real data is to use DESeq2 or edgeR for the actual differential expression step since they handle dispersion estimation much better than a simple t-test, and I'd add a gene ontology enrichment step to figure out which biological processes are being affected. But as a starting point for exploratory work, this pipeline is clean and gets you oriented quickly.

---
*Analysis performed in Python 3.12 | Libraries: NumPy, Pandas, SciPy, Scikit-learn, Matplotlib, Seaborn, Statsmodels*